### 앙상블(Ensemble)
- 여러개의 분류 모델을 조합해서 더 나은 성능을 내는 방법
- Decision Tree에서 모델을 증가시켜 나온 랜덤포레스트가 대표적임 

#### 랜덤포레스트
- 부트스트랩 샘플 : 중복을 허용하는 샘플링 방법 
- 샘플링 후에 샘플을 복구하고 다시 샘플링 하는 방법
- 이와 같이 진행하는 이유는 결정트리에서 과대적합을 방지 할 수 있기 때문 
- 각 결정트리에서 나오는 확률의 합을 트리갯수로 나누어 결정하는 모델
- 특성의 갯수를 제곱근하여 부트 샘플링할 갯수를 결정한다.  

In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [2]:
wine = pd.read_csv("../Data/wine.csv")
wine.head()

,alcohol,sugar,pH,class
0,9.4,1.9,3.51,0.0
1,9.8,2.6,3.20,0.0
2,9.8,2.3,3.26,0.0
3,9.8,1.9,3.16,0.0
4,9.4,1.9,3.51,0.0


In [3]:
# Feature와 Target
data = wine.iloc[:,:3].to_numpy()
target = wine.iloc[:,3].to_numpy()

In [4]:
# 전체 세트중 훈련세트와 테스트세트를 8:2의 기준으로 분리한다. 
from sklearn.model_selection import train_test_split

In [5]:
train_input, test_input, train_target, test_target = \
                        train_test_split(
                                data,
                                target,
                                test_size=0.2,
                                random_state=42
                        )

#### 랜덤포레스트

In [6]:
from sklearn.model_selection import cross_validate
from sklearn.ensemble import RandomForestClassifier

In [7]:
rf = RandomForestClassifier(n_jobs=-1, random_state=42)
scores = cross_validate(
            rf,
            train_input,
            train_target,
            n_jobs=-1,
            return_train_score=True
)
scores

{'fit_time': array([0.24792886, 0.25294185, 0.25609517, 0.24946785, 0.28283334]),
 'score_time': array([0.04285669, 0.04036188, 0.03432369, 0.02882123, 0.02670002]),
 'test_score': array([0.88461538, 0.88942308, 0.90279115, 0.88931665, 0.88642926]),
 'train_score': array([0.9971133 , 0.99663219, 0.9978355 , 0.9973545 , 0.9978355 ])}

In [8]:
print("Train :", scores['train_score'].mean())
print("Test :", scores['test_score'].mean())

Train : 0.9973541965122431
Test : 0.8905151032797809


In [10]:
# 주요 Feature 확인
rf.fit(train_input, train_target)
rf.feature_importances_

array([0.23167441, 0.50039841, 0.26792718])

In [11]:
# 부트스트랩 결정시 남는 샘플(oob:out of back)로 특성을 구분할 수 있다. 
rf = RandomForestClassifier(
                oob_score=True,
                n_jobs=-1,
                random_state=42
)

rf.fit(train_input, train_target)
print(rf.oob_score_)

0.8934000384837406


따로 검증셋을 구성하지 않아도 oob로 검증 세트 역할을 대신할 수 있다.   

----
#### Extra Tree
- 기본적으로 100개의 트리를 사용
- 노드 분할시 특성의 제곱근의 갯수를 사용
- 특성의 선택을 랜덤하게 선택한다
- 특성의 선택을 랜덤하게 하므로 속도는 랜덤포레스트보다 빠르다. 

In [12]:
from sklearn.ensemble import ExtraTreesClassifier
et = ExtraTreesClassifier(n_jobs=-1, random_state=42)
scores = cross_validate(
                et,
                train_input,
                train_target,
                return_train_score=True,
                n_jobs=-1
)
print(scores['train_score'].mean(), scores['test_score'].mean())

0.9974503966084433 0.8887848893166506


----
#### Gradient Boosting
- 가장 유명한 알고리즘중 하나이다. 
- 경사하강법 처럼 손실함수를 사용.
- 손실함수를 보고 트리를 추가하여 최적의 값을 도출하는 방법
- Decision Tree Regressor를 사용하여 손실함수 계산하고 이를 낮추기 위해 트리를 추가하는 구조
- max_depth를 3으로 제어하여 깊이가 낮으므로 과대적합 방지
- 단점은 손실함수를 확인하고 트리를 추가하면서 진행하는 모델이므로 n_jobs(병렬처리)를 할 수 없다. 

In [13]:
from sklearn.ensemble import GradientBoostingClassifier
gb = GradientBoostingClassifier(random_state=42)
scores = cross_validate(
                gb,
                train_input,
                train_target,
                return_train_score=True,
                n_jobs=-1
)
print(scores['train_score'].mean(), scores['test_score'].mean())

0.8881086892152563 0.8720430147331015


---
#### Histogram Gradient Boosting
- 훈련데이터를 256개의 구간으로 나누어서 훈련시키는 방법
- 특성의 범위가 제한되어 있어 빠른 속도를 제공
- 제한된 구간이므로 과대적합을 방지


In [14]:
from sklearn.ensemble import HistGradientBoostingClassifier
hgb = HistGradientBoostingClassifier(random_state=42)
scores = cross_validate(
                hgb,
                train_input,
                train_target,
                return_train_score=True,
                n_jobs=-1
)
print(scores['train_score'].mean(), scores['test_score'].mean())

0.9321723946453317 0.8801241948619236


---
#### XGBoost
- Kaggle에서 많이 사용

In [ ]:
# !pip install xgboost

   ---------------------------------------- 0.0/150.0 MB ? eta -:--:--
   ---------------------------------------- 1.3/150.0 MB 9.5 MB/s eta 0:00:16
   - -------------------------------------- 5.5/150.0 MB 16.0 MB/s eta 0:00:10
   --- ------------------------------------ 11.5/150.0 MB 20.0 MB/s eta 0:00:07
   ---- ----------------------------------- 17.0/150.0 MB 21.9 MB/s eta 0:00:07
   ----- ---------------------------------- 21.8/150.0 MB 21.8 MB/s eta 0:00:06
   ------- -------------------------------- 26.5/150.0 MB 21.8 MB/s eta 0:00:06
   -------- ------------------------------- 31.5/150.0 MB 21.9 MB/s eta 0:00:06
   --------- ------------------------------ 36.4/150.0 MB 22.0 MB/s eta 0:00:06
   ----------- ---------------------------- 41.4/150.0 MB 22.1 MB/s eta 0:00:05
   ------------ --------------------------- 46.4/150.0 MB 22.2 MB/s eta 0:00:05
   ------------- -------------------------- 51.1/150.0 MB 22.3 MB/s eta 0:00:05
   -------------- ------------------------- 54.3/150

In [17]:
from xgboost import XGBClassifier

In [18]:
xgb = XGBClassifier(
            tree_method = 'hist',
            random_state = 42,
            eval_metric = 'logloss',
            use_label_encoder = False
)

In [19]:
scores = cross_validate(
                xgb,
                train_input,
                train_target,
                return_train_score=True,
                n_jobs=-1
)
print(scores['train_score'].mean(), scores['test_score'].mean())

0.9567059184812372 0.8783915747390243


----
#### LightGBM
: Gradient Boosting에서 출발

In [21]:
# !pip install lightgbm

In [22]:
from lightgbm import LGBMClassifier

In [31]:
lgb = LGBMClassifier(random_state=42, force_col_wise=True)
lgb

LGBMClassifier(force_col_wise=True, random_state=42)

In [ ]:
scores = cross_validate(
                lgb,
                train_input,
                train_target,
                return_train_score=True,
                n_jobs=-1
)
print(scores['train_score'].mean(), scores['test_score'].mean())

0.935828414851749 0.8801251203079884


In [33]:
lgb.fit(train_input, train_target)

LGBMClassifier(force_col_wise=True, random_state=42)

In [34]:
lgb.score(test_input, test_target)

0.8730769230769231

----
#### Permutation Importance(치환 중요도)
- 각 특성별 sample을 섞어서 계산을 한후에 원래 sample들과의 차이를 계산하여 차이가 많이 나는 Feature가 중요하다는 판단을 한다. 
- 즉, 어떤 Feature가 중요한지 파악하느 방법 (EDA)
- 어떤 모델에도 사용가능하며 특성을 파악하는 주요 기준으로 사용. 권장 사항임(EDA)

In [35]:
# Train의 경우
from sklearn.inspection import permutation_importance

In [36]:
hgb.fit(train_input, train_target)
result = permutation_importance(
                hgb,
                train_input,
                train_target,
                n_repeats=10,
                random_state=42,
                n_jobs=-1
)

In [38]:
result['importances_mean']

array([0.08876275, 0.23438522, 0.08027708])

> Sugar인 경우 Feature를 무작위로 섞으면 23%만큼 정확도가 떨어진다.  